# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 8: The Quantum Threat

**60 minutes taught · 90 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and preparation

Distinguish the threats to public-key mechanisms from those to symmetric primitives. Explain harvest-now-decrypt-later (HNDL), and prioritize a migration using data lifetime and migration lead time without inventing a quantum-computer arrival date. Use the Day 2 setup (see course website). This notebook is a planning model, not a quantum simulator or a cryptanalytic benchmark.

## Which assumptions change?

Shor's algorithm gives a quantum approach to factoring and discrete logarithms. A sufficiently capable, fault-tolerant quantum computer could therefore undermine RSA and the elliptic-curve mechanisms used earlier. Today's existence of small experimental quantum machines is not evidence that they can break deployed key sizes. Logical error correction, scale, time and engineering resources matter.

Grover's algorithm offers a generic quadratic improvement in unstructured search in the ideal query model. It is not a statement that an attacker gets a free, practical halving of every system's security. Circuit cost, parallelism, time limits and target structure affect real attacks. AES and hashes do not simply disappear; suitable parameters and sound protocols remain important. Larger RSA keys are not a post-quantum replacement for RSA's vulnerable mathematical structure.

| Mechanism | Quantum concern | Engineering consequence |
| --- | --- | --- |
| RSA, X25519, ECDSA, Ed25519 | Factoring or discrete-log assumptions | Plan replacement of affected key establishment and signatures |
| AES | Generic key search improvement in an ideal quantum model | Retain symmetric encryption with an appropriate strength and usage policy |
| Hashes / HKDF | Different quantum query effects and construction assumptions | Review parameters and construction, not just output length |
| Password verifiers | Low-entropy guesses remain a problem today | PQ migration does not repair weak passwords or credential theft |

## Harvest now, decrypt later

```mermaid
flowchart LR
    T["Today: capture public handshake and encrypted records"] --> S["Store for years"]
    S --> Q["Future capability recovers classical exchange secret"]
    Q --> K["Reconstruct traffic keys from recorded public context"]
    K --> D["Decrypt historical confidential records"]
```

The attacker need not compromise today's endpoints if the later cryptanalysis reconstructs the recorded exchange secret. Classical forward secrecy protects against a different event: later theft of a long-term key under classical assumptions. It does not make classical DH immune to Shor's algorithm.

Changing algorithms later does not remove an adversary's earlier recording. Retention of highly confidential state, research, diplomatic or personal data can make action useful before the future capability exists. PQC runs on ordinary computers; it is not quantum key distribution and does not require a quantum network.

## A planning inequality, not a forecast

Let `L` be the remaining secrecy lifetime in years, `M` the years required to migrate, and `H` an assumed planning horizon until a relevant adversary capability. If `L + M > H`, waiting is inconsistent with that scenario's secrecy objective. This heuristic exposes assumptions; it is not a probability model or proof of safety when the inequality is false.


In [ ]:
assets = [
    {'name': 'highly confidential archive', 'L': 20, 'M': 5},
    {'name': 'research dataset', 'L': 10, 'M': 3},
    {'name': 'short-lived telemetry', 'L': 1, 'M': 2},
]
def planning_gap(lifetime, migration, horizon):
    if min(lifetime, migration, horizon) < 0:
        raise ValueError('Use nonnegative years')
    return lifetime + migration - horizon
for asset in assets:
    gaps = {h: planning_gap(asset['L'], asset['M'], h) for h in (5, 10, 20)}
    print(asset['name'], gaps)
assert planning_gap(20, 5, 10) == 15
assert planning_gap(1, 2, 10) < 0
print('PASS: scenario arithmetic uses explicit hypothetical horizons, not arrival predictions')


Positive values indicate a gap in the chosen scenario. Negative values do not address endpoint compromise, existing collection, cost of failure, or uncertainty in the assumptions.


In [ ]:
import matplotlib.pyplot as plt
horizons = [5, 10, 15, 20, 25, 30]
fig, ax = plt.subplots(figsize=(8, 4))
for asset in assets:
    ax.plot(horizons, [planning_gap(asset['L'], asset['M'], h) for h in horizons],
            marker='o', label=asset['name'])
ax.axhline(0, color='black', linewidth=1)
ax.set(xlabel='Assumed capability horizon from now (years)',
       ylabel='L + M - H (years)', title='Hypothetical planning scenarios — not a forecast')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()
print('PASS: plotted sensitivity to explicitly assumed horizons')


Change `L` and `M` to observe which decisions are robust across assumptions. Do not label the x-axis “the year quantum computers arrive.”

```mermaid
xychart-beta
    title "Archive scenario: L = 20 years, M = 5 years"
    x-axis "Assumed horizon H in years, not a forecast" [5, 10, 15, 20, 25, 30]
    y-axis "L + M - H in years" -5 --> 20
    line [20, 15, 10, 5, 0, -5]
```

This static view shows one scenario from the notebook. Crossing zero changes this heuristic's result, not a guarantee about the archive's safety.

## Inventory the dependency, not only the algorithm

```mermaid
flowchart TD
    D["Asset and secrecy/authenticity lifetime"] --> P["Where is public-key protection used?"]
    P --> T["Transport and termination"]
    P --> S["Stored key wrapping and backups"]
    P --> A["Signing, trust anchors and update systems"]
    T --> O["Owner, dependency, upgrade path and evidence"]
    S --> O
    A --> O
```

Record what protects the data-encryption key, who can change it, and whether old wrapped keys or transcripts remain recoverable. Rewrapping stored data helps only under an analyzed architecture and cannot undo copies already captured. Signature migration has different consequences: future forgery, trust-anchor replacement and long-term verification evidence rather than merely decrypting recordings.

For a state actor, also model insider access, supplier compromise, coercion of operators and endpoint capture. Post-quantum algorithms address a mathematical threat; they do not eliminate those paths.

## Exercise and worked decision

An archive must remain confidential for 25 years. Its TLS gateway can be upgraded in one year, but its backup key-wrapping system and offline readers require six years. Make separate inventory entries, owners and milestones. Do not describe the whole archive as migrated after replacing the gateway.

<details><summary>Worked reasoning and exit answers</summary>
<p>The slow backup/reader dependency governs one major exposure path. Assess recordings and stored wrapped keys separately, test interoperability, and plan trust changes. No single forecast is required to see long-lived exposure. AES-256 records protected by a recoverable classical exchange are still exposed through the recovered traffic key; classical forward secrecy does not change that conclusion. A short-lived asset may be lower priority for confidentiality but still require long-lived signature validation or upgrade capability.</p>
</details>

Ready to continue: explain why “we use AES-256” and “we have forward secrecy” do not complete an HNDL assessment. Next: ML-KEM (see course website).

## Sources

Reviewed 22 September 2026: [NIST PQC FAQs](https://csrc.nist.gov/projects/post-quantum-cryptography/faqs). Scenario values are invented teaching inputs, not estimates endorsed by NIST.


In [ ]:
print("PASS: completed session-08-quantum-threat demonstrations; learner status is reported separately")
